*❗❗ Before you run this lab, go to Runtime → Change Runtime Type → Choose: T4 GPU*

# Lab 5: CV Metrics & Demographic Bias Audit
**AIML 2013 — Computer Vision**

*Measure a classifier's accuracy, then find out who it works for and who it doesn't.*

---

This is the standalone CV lab for Module 5. You will work in one notebook, give one 3–5 minute demo, and submit your GitHub repo link to Canvas.


## How This Lab Works

In the readings and in class, you learned that a model's overall accuracy can mask large disparities across subgroups. A face-attribute classifier that is 94% accurate overall might be 99% accurate on one subgroup and 65% accurate on another. The overall number looks good; the per-group numbers tell a different story.

This lab uses a pre-trained image classifier and a dataset with demographic labels to run a structured bias audit. You will compute standard classification metrics (accuracy, precision, recall, F1) on the full dataset, then disaggregate those metrics by subgroup to find where the model fails disproportionately.

The lab is less about code and more about what the numbers reveal — the same classifier, the same test set, but a completely different conclusion depending on how you slice the results.

**Dataset.** 1,000 images from the CelebA test set loaded via `tensorflow_datasets`. Each image has 40 binary attribute labels. We predict `Male` and audit by `Young`, `Wearing_Hat`, `Smiling`, and `Eyeglasses`.

**Classifier.** MobileNetV2 as a frozen feature extractor, with a logistic regression head trained on 2,000 CelebA training images. Wrapped as a single Keras model so `model(x)` returns P(Male).


---
## Part 1: Setup and Baseline Metrics


### Cell 1: Setup

Installs dependencies, imports packages, and sets random seeds.


In [ ]:
# Cell 1: Setup
!pip install -q tensorflow_datasets

import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow {tf.__version__}")
print(f"TFDS       {tfds.__version__}")
print("GPU available:", bool(tf.config.list_physical_devices("GPU")))
print("Setup complete.")


### Cell 2: Load Dataset and Classifier

Loads 2,000 training images (for fitting the classifier head) and 1,000 test images (for the audit). Extracts MobileNetV2 embeddings for both, fits a logistic-regression head on the training embeddings, then copies the fitted weights into a Keras `Dense(1, sigmoid)` layer. The final `model` is a single Keras model that takes a preprocessed image batch and returns `P(Male)`.

If the `tensorflow_datasets` download fails (Google Drive rate limits), you can instead use Hugging Face: `datasets.load_dataset("student/celebA")` or manually download via `!gdown 0B7EVK8r0v71pblRyaVFSWGxPY0U -O celeba.zip`.


In [ ]:
# Cell 2: Load dataset + build classifier

# ------------------------------------------------------------
# Load CelebA via tensorflow_datasets
# ------------------------------------------------------------
print("Loading CelebA test[:1000] and train[:2000]...")
test_ds = tfds.load("celeb_a", split="test[:1000]")
train_ds = tfds.load("celeb_a", split="train[:2000]")

ATTR_NAMES = ["Male", "Young", "Wearing_Hat", "Smiling", "Eyeglasses"]

def to_numpy(ds):
    """Resize images to 224x224 and pull out the attribute dict."""
    imgs, attrs = [], []
    for ex in tfds.as_numpy(ds):
        img = tf.image.resize(ex["image"], (224, 224)).numpy().astype(np.float32)
        imgs.append(img)
        attrs.append({k: bool(ex["attributes"][k]) for k in ATTR_NAMES})
    return np.array(imgs), attrs

test_images_raw, test_attrs = to_numpy(test_ds)
train_images_raw, train_attrs = to_numpy(train_ds)
print(f"Test:  {test_images_raw.shape}")
print(f"Train: {train_images_raw.shape}")

# ------------------------------------------------------------
# Preprocess for MobileNetV2 (keep the raw uint8-range copy for display)
# ------------------------------------------------------------
images = tf.keras.applications.mobilenet_v2.preprocess_input(test_images_raw.copy())
train_images = tf.keras.applications.mobilenet_v2.preprocess_input(train_images_raw.copy())

# ------------------------------------------------------------
# Build labels
# ------------------------------------------------------------
labels_df = pd.DataFrame(test_attrs)
y_true = labels_df["Male"].astype(int).values
y_train = np.array([a["Male"] for a in train_attrs]).astype(int)

# ------------------------------------------------------------
# MobileNetV2 feature extractor
# ------------------------------------------------------------
print("Extracting MobileNetV2 embeddings...")
base = tf.keras.applications.MobileNetV2(
    include_top=False, weights="imagenet", pooling="avg",
)
base.trainable = False

train_emb = base.predict(train_images, verbose=1)
test_emb = base.predict(images, verbose=1)

# ------------------------------------------------------------
# Fit a logistic-regression head, then bake it into a Keras Dense layer
# ------------------------------------------------------------
print("Fitting logistic regression head...")
lr = LogisticRegression(max_iter=2000, random_state=42)
lr.fit(train_emb, y_train)
print(f"  Train accuracy: {lr.score(train_emb, y_train):.3f}")

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base(inputs, training=False)
outputs = tf.keras.layers.Dense(1, activation="sigmoid", name="gender_head")(x)
model = tf.keras.Model(inputs, outputs, name="MobileNetV2_gender")

# Copy LR weights into the Dense head: sklearn LR gives coef_ shape (1, 1280)
# and Keras Dense expects (1280, 1). Intercept_ is shape (1,).
W = lr.coef_.T.astype(np.float32)
b = lr.intercept_.astype(np.float32)
model.get_layer("gender_head").set_weights([W, b])

# ------------------------------------------------------------
# Quick sanity summary + sample grid
# ------------------------------------------------------------
n_male = int(y_true.sum())
n_not_male = int((1 - y_true).sum())
print(f"\nTest images: {len(images)}")
print(f"  Male:     {n_male}  ({n_male / len(images):.1%})")
print(f"  Not Male: {n_not_male}  ({n_not_male / len(images):.1%})")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, img, label in zip(axes.flat, test_images_raw[:8], y_true[:8]):
    ax.imshow(img.astype(np.uint8))
    ax.set_title("Male" if label else "Not Male", fontsize=11)
    ax.axis("off")
fig.suptitle("Sample CelebA test images", fontsize=13)
plt.tight_layout()
plt.show()


**✍️ Reflection.** What is the gender balance in your test set? If the test set is 60% Male, what does that mean for interpreting the model's accuracy? (A dumb classifier that always predicts "Male" would get 60% on that test set — overall accuracy alone hides that.)


### Cell 3: Overall Predictions and Metrics

Runs the classifier across all 1,000 test images and reports accuracy / precision / recall / F1, a full classification report, and the confusion matrix.


In [ ]:
# Cell 3: Overall predictions and metrics

y_pred_proba = model.predict(images, verbose=1).ravel()
y_pred = (y_pred_proba >= 0.5).astype(int)

overall_metrics = {
    "accuracy":  accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred),
    "recall":    recall_score(y_true, y_pred),
    "f1":        f1_score(y_true, y_pred),
}

print("=== Overall metrics ===")
for k, v in overall_metrics.items():
    print(f"  {k:<10} {v:.4f}")

print("\n=== Classification report ===")
print(classification_report(y_true, y_pred, target_names=["Not Male", "Male"]))

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=["Not Male", "Male"],
    yticklabels=["Not Male", "Male"],
    cbar=False, ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Overall confusion matrix  (acc = {overall_metrics['accuracy']:.3f})")
plt.tight_layout()
plt.show()


**✍️ Reflection.** Look at the confusion matrix. Are false positives and false negatives roughly balanced, or does the model make one type of error more than the other? What might cause that asymmetry? (Hint: class imbalance in the training set, or systematic feature differences that push predictions toward one class.)


---
## Part 2: The Demographic Audit

Overall metrics summarize a classifier in one number per dimension. The audit in this section disaggregates those numbers by subgroup and asks: *for whom does this model work, and for whom does it fail?*


### Cell 4: Per-Group Accuracy by Attribute

For each slicing attribute (`Young`, `Wearing_Hat`, `Smiling`, `Eyeglasses`), computes accuracy, precision, recall, F1, and sample size separately for the True and False subgroups. Shows a grouped bar chart with the overall accuracy drawn as a reference line.


In [ ]:
# Cell 4: Per-group accuracy by attribute

slice_attributes = ["Young", "Wearing_Hat", "Smiling", "Eyeglasses"]

rows = []
for attr in slice_attributes:
    for value in [True, False]:
        mask = labels_df[attr].values == value
        n = int(mask.sum())
        if n == 0:
            continue
        yt = y_true[mask]
        yp = y_pred[mask]
        rows.append({
            "attribute": attr,
            "group":     value,
            "accuracy":  accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall":    recall_score(yt, yp, zero_division=0),
            "f1":        f1_score(yt, yp, zero_division=0),
            "n_samples": n,
        })

group_metrics = pd.DataFrame(rows)
print(group_metrics.to_string(index=False))

# ------------------------------------------------------------
# Grouped bar chart
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 6))
x_positions, tick_labels, colors, accs = [], [], [], []
for i, attr in enumerate(slice_attributes):
    for j, val in enumerate([True, False]):
        row = group_metrics[(group_metrics["attribute"] == attr) &
                            (group_metrics["group"] == val)].iloc[0]
        x_positions.append(i * 2.6 + j)
        tick_labels.append(f"{attr}={val}\n(n={row['n_samples']})")
        colors.append("#2ca02c" if val else "#d62728")
        accs.append(row["accuracy"])

bars = ax.bar(x_positions, accs, color=colors, alpha=0.8, edgecolor="black")
ax.axhline(
    overall_metrics["accuracy"], linestyle="--", color="black",
    label=f"Overall = {overall_metrics['accuracy']:.3f}",
)
ax.set_xticks(x_positions)
ax.set_xticklabels(tick_labels, fontsize=9)
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1.08)
ax.set_title("Gender-classification accuracy, sliced by attribute")
ax.legend(loc="lower right")

for bar, acc in zip(bars, accs):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.012,
        f"{acc:.3f}", ha="center", fontsize=9,
    )

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Small-sample warnings + best/worst callout
# ------------------------------------------------------------
small = group_metrics[group_metrics["n_samples"] < 20]
if len(small):
    print("\nCaution: small sample size in these groups (<20):")
    for _, r in small.iterrows():
        print(f"  {r['attribute']}={r['group']}: n={r['n_samples']}")

best_row = group_metrics.loc[group_metrics["accuracy"].idxmax()]
worst_row = group_metrics.loc[group_metrics["accuracy"].idxmin()]
print(f"\nHighest accuracy: {best_row['attribute']}={best_row['group']}  acc={best_row['accuracy']:.3f}  n={best_row['n_samples']}")
print(f"Lowest  accuracy: {worst_row['attribute']}={worst_row['group']}  acc={worst_row['accuracy']:.3f}  n={worst_row['n_samples']}")
print(f"Gap: {best_row['accuracy'] - worst_row['accuracy']:.3f}")


**✍️ Reflection.** Which subgroup has the lowest accuracy? Is the gap between the best and worst subgroup larger than you expected? What might explain the difference — training-set representation, visual distinctiveness of the attribute, something else?


### Cell 5: Intersectional Analysis

Overall accuracy hides per-group gaps. Per-group accuracy can hide *intersectional* gaps: a subgroup that looks fine on attribute A and fine on attribute B may still have low accuracy when both co-occur.

Here we cross `Young × Smiling` and measure accuracy for all four combinations.


In [ ]:
# Cell 5: Intersectional analysis

ATTR_A = "Young"
ATTR_B = "Smiling"

rows = []
for a_val in [True, False]:
    for b_val in [True, False]:
        mask = (labels_df[ATTR_A].values == a_val) & (labels_df[ATTR_B].values == b_val)
        n = int(mask.sum())
        if n == 0:
            rows.append({"group_a": a_val, "group_b": b_val,
                         "accuracy": np.nan, "precision": np.nan,
                         "recall": np.nan, "f1": np.nan, "n_samples": 0})
            continue
        yt = y_true[mask]
        yp = y_pred[mask]
        rows.append({
            "group_a": a_val, "group_b": b_val,
            "accuracy":  accuracy_score(yt, yp),
            "precision": precision_score(yt, yp, zero_division=0),
            "recall":    recall_score(yt, yp, zero_division=0),
            "f1":        f1_score(yt, yp, zero_division=0),
            "n_samples": n,
        })

intersect_metrics = pd.DataFrame(rows)
intersect_metrics.insert(0, "attribute_a", ATTR_A)
intersect_metrics.insert(1, "attribute_b", ATTR_B)
print(intersect_metrics.to_string(index=False))

# ------------------------------------------------------------
# Build a 2x2 heatmap matrix
# ------------------------------------------------------------
acc_matrix = np.full((2, 2), np.nan)
annot = np.empty((2, 2), dtype=object)
for _, r in intersect_metrics.iterrows():
    i = 0 if r["group_a"] else 1   # row 0 = True, row 1 = False
    j = 0 if r["group_b"] else 1   # col 0 = True, col 1 = False
    acc_matrix[i, j] = r["accuracy"]
    if pd.notna(r["accuracy"]):
        annot[i, j] = f"acc={r['accuracy']:.3f}\nn={r['n_samples']}"
    else:
        annot[i, j] = "no samples"

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(
    acc_matrix, annot=annot, fmt="", cmap="RdYlGn", vmin=0.5, vmax=1.0,
    xticklabels=[f"{ATTR_B}=True", f"{ATTR_B}=False"],
    yticklabels=[f"{ATTR_A}=True", f"{ATTR_A}=False"],
    cbar_kws={"label": "Accuracy"}, ax=ax,
)
ax.set_title(f"Intersectional accuracy: {ATTR_A} \u00d7 {ATTR_B}")
plt.tight_layout()
plt.show()

gap = intersect_metrics["accuracy"].max() - intersect_metrics["accuracy"].min()
print(f"\nAccuracy gap best - worst intersection: {gap:.3f}")

small = intersect_metrics[intersect_metrics["n_samples"] < 15]
if len(small):
    print("\nCaution: intersections with <15 samples — interpret with care:")
    for _, r in small.iterrows():
        print(f"  {ATTR_A}={r['group_a']}, {ATTR_B}={r['group_b']}: n={r['n_samples']}")


**✍️ Reflection.** Did the intersectional analysis reveal a subgroup that performs worse than either individual attribute suggested? This is the core insight of intersectional auditing: combined identities can produce failures that single-attribute analysis misses.


### Cell 6: Per-Group Confusion Matrices

Side-by-side confusion matrices for the best-performing and worst-performing subgroups. Normalized by row so error rates are comparable even when sample sizes differ.


In [ ]:
# Cell 6: Per-group confusion matrices

best = group_metrics.loc[group_metrics["accuracy"].idxmax()]
worst = group_metrics.loc[group_metrics["accuracy"].idxmin()]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

def plot_group_cm(ax, row, title):
    mask = labels_df[row["attribute"]].values == row["group"]
    yt = y_true[mask]
    yp = y_pred[mask]
    cm_raw = confusion_matrix(yt, yp, labels=[0, 1])
    # Normalize per row (safe divide)
    row_sums = cm_raw.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm_raw, row_sums, out=np.zeros_like(cm_raw, dtype=float),
                        where=row_sums > 0)
    annot = np.array([
        [f"{cm_raw[i, j]}\n({cm_norm[i, j]:.1%})" for j in range(2)]
        for i in range(2)
    ])
    sns.heatmap(
        cm_norm, annot=annot, fmt="", cmap="Blues", vmin=0, vmax=1,
        xticklabels=["Not Male", "Male"],
        yticklabels=["Not Male", "Male"],
        cbar=False, ax=ax,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(
        f"{title}: {row['attribute']}={row['group']}\n"
        f"acc={row['accuracy']:.3f}, n={row['n_samples']}"
    )
    return cm_raw

cm_best  = plot_group_cm(axes[0], best,  "Best")
cm_worst = plot_group_cm(axes[1], worst, "Worst")

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# False-negative-rate comparison
# (actual Male but predicted Not Male)
# ------------------------------------------------------------
def fnr(cm):
    actual_pos = cm[1].sum()
    return cm[1, 0] / actual_pos if actual_pos > 0 else 0.0

def fpr(cm):
    actual_neg = cm[0].sum()
    return cm[0, 1] / actual_neg if actual_neg > 0 else 0.0

print(
    f"\nFalse negative rate on worst group "
    f"({worst['attribute']}={worst['group']}): {fnr(cm_worst):.1%}"
)
print(
    f"False negative rate on best  group "
    f"({best['attribute']}={best['group']}):  {fnr(cm_best):.1%}"
)
print(
    f"False positive rate: worst {fpr(cm_worst):.1%}  |  best {fpr(cm_best):.1%}"
)


**✍️ Reflection.** Compare the false negative rates. In a real deployment (identity verification, photo tagging), which error is more harmful — a false positive or a false negative? Does the answer change depending on the application?


---
## Part 3: The Audit Report

The audit is only useful if its findings land. The two cells in this section are the artifact you will actually show in your demo: a one-page scorecard and a gallery of real failures.


### Cell 7: Bias Scorecard

A four-panel figure that summarizes the audit:

1. Overall metrics bar chart.
2. Per-attribute accuracy gap (green < 5%, yellow 5–10%, red > 10%).
3. The intersectional heatmap from Cell 5.
4. A text-summary panel with an audit verdict.


In [ ]:
# Cell 7: Bias scorecard

fig, axes = plt.subplots(2, 2, figsize=(15, 11))

# ------------------------------------------------------------
# Panel 1: overall metrics
# ------------------------------------------------------------
ax1 = axes[0, 0]
metric_names = list(overall_metrics.keys())
metric_values = [overall_metrics[k] for k in metric_names]
bars1 = ax1.bar(metric_names, metric_values, color="#1a3a5c")
ax1.set_ylim(0, 1.08)
ax1.set_title("Overall metrics", fontsize=13, fontweight="bold")
for bar, v in zip(bars1, metric_values):
    ax1.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.015,
        f"{v:.3f}", ha="center", fontsize=11,
    )

# ------------------------------------------------------------
# Panel 2: per-attribute accuracy gap
# ------------------------------------------------------------
ax2 = axes[0, 1]
gap_rows = []
for attr in slice_attributes:
    rows_attr = group_metrics[group_metrics["attribute"] == attr]
    gap = rows_attr["accuracy"].max() - rows_attr["accuracy"].min()
    gap_rows.append((attr, gap))

gap_labels = [g[0] for g in gap_rows]
gap_vals   = [g[1] for g in gap_rows]
gap_colors = [
    "#2ca02c" if g < 0.05 else "#f9a825" if g < 0.10 else "#d62728"
    for g in gap_vals
]
bars2 = ax2.barh(gap_labels, gap_vals, color=gap_colors, edgecolor="black")
ax2.axvline(0.05, linestyle=":", color="gray", alpha=0.6)
ax2.axvline(0.10, linestyle=":", color="gray", alpha=0.6)
ax2.set_xlabel("Accuracy gap (best − worst)")
ax2.set_title("Per-attribute gap  (green<5%  yellow<10%  red≥10%)",
              fontsize=13, fontweight="bold")
for bar, g in zip(bars2, gap_vals):
    ax2.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height() / 2,
             f"{g:.3f}", va="center", fontsize=10)

# ------------------------------------------------------------
# Panel 3: intersectional heatmap (reuses acc_matrix / annot from Cell 5)
# ------------------------------------------------------------
ax3 = axes[1, 0]
sns.heatmap(
    acc_matrix, annot=annot, fmt="", cmap="RdYlGn", vmin=0.5, vmax=1.0,
    xticklabels=[f"{ATTR_B}=True", f"{ATTR_B}=False"],
    yticklabels=[f"{ATTR_A}=True", f"{ATTR_A}=False"],
    cbar_kws={"label": "Accuracy"}, ax=ax3,
)
ax3.set_title(f"Intersectional: {ATTR_A} \u00d7 {ATTR_B}",
              fontsize=13, fontweight="bold")

# ------------------------------------------------------------
# Panel 4: summary panel
# ------------------------------------------------------------
ax4 = axes[1, 1]
ax4.axis("off")

largest_gap = max(gap_vals)
largest_gap_attr = gap_labels[int(np.argmax(gap_vals))]
worst_row_overall = group_metrics.loc[group_metrics["accuracy"].idxmin()]

if largest_gap < 0.05:
    verdict = "No significant disparities detected."
    verdict_color = "#2ca02c"
elif largest_gap < 0.10:
    verdict = f"Moderate disparities detected in {largest_gap_attr}."
    verdict_color = "#f9a825"
else:
    verdict = f"Significant disparities detected in {largest_gap_attr}."
    verdict_color = "#d62728"

summary = (
    "AUDIT SUMMARY\n"
    "--------------\n"
    f"Overall accuracy:     {overall_metrics['accuracy']:.3f}\n"
    f"Largest per-group gap: {largest_gap:.3f}  ({largest_gap_attr})\n"
    f"Worst subgroup:       {worst_row_overall['attribute']}={worst_row_overall['group']}\n"
    f"                      acc={worst_row_overall['accuracy']:.3f}, "
    f"n={worst_row_overall['n_samples']}\n\n"
    f"VERDICT:\n{verdict}"
)

ax4.text(
    0.03, 0.97, summary, transform=ax4.transAxes, fontsize=12,
    verticalalignment="top", family="monospace",
    bbox=dict(boxstyle="round,pad=0.8", facecolor="#f7f9fc",
              edgecolor=verdict_color, linewidth=2),
)

fig.suptitle(
    "Bias Audit Scorecard: Gender Classification on CelebA",
    fontsize=16, fontweight="bold",
)
plt.tight_layout()
plt.show()


**✍️ Reflection.** Look at your scorecard. If you were presenting this to a product team deciding whether to deploy this classifier, what would be your one-sentence recommendation? Would you approve deployment? Under what conditions?


### Cell 8: Failure Gallery

Numbers describe the problem; images make it concrete. This cell pulls misclassified images from the worst-performing subgroup so we can look at them directly.


In [ ]:
# Cell 8: Failure gallery

worst_row_overall = group_metrics.loc[group_metrics["accuracy"].idxmin()]
mask = labels_df[worst_row_overall["attribute"]].values == worst_row_overall["group"]
worst_indices = np.where(mask)[0]
error_mask = y_pred[worst_indices] != y_true[worst_indices]
error_indices = worst_indices[error_mask]

print(f"Worst subgroup:       {worst_row_overall['attribute']}={worst_row_overall['group']}")
print(f"Subgroup size:        {len(worst_indices)}")
print(f"Total misclassified:  {len(error_indices)}")

to_show = error_indices[:12]
if len(to_show) == 0:
    print("\nNo misclassifications in the worst subgroup — nothing to show.")
else:
    cols = 4
    rows_grid = int(np.ceil(len(to_show) / cols))
    fig, axes = plt.subplots(rows_grid, cols, figsize=(14, 3.8 * rows_grid))
    axes = np.atleast_2d(axes)

    for idx, ax in zip(to_show, axes.flat):
        # test_images_raw is in float32 but in original pixel range (0-255)
        ax.imshow(test_images_raw[idx].astype(np.uint8))
        true_label = "Male" if y_true[idx] else "Not Male"
        pred_label = "Male" if y_pred[idx] else "Not Male"
        p = y_pred_proba[idx]
        confidence = p if y_pred[idx] else 1 - p
        ax.set_title(
            f"True: {true_label}  |  Pred: {pred_label}\nconfidence={confidence:.2f}",
            fontsize=10, color="#b71c1c",
        )
        ax.axis("off")

    # Hide any unused axes
    for extra_ax in axes.flat[len(to_show):]:
        extra_ax.axis("off")

    fig.suptitle(
        f"Misclassifications in worst subgroup: "
        f"{worst_row_overall['attribute']}={worst_row_overall['group']}",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()


**✍️ Reflection.** Look at the misclassified images. Can you see any visual patterns? Are the errors random, or do they cluster on images with specific characteristics — lighting, pose, accessories, angle?


---
## Bonus: Threshold Sensitivity Analysis *(10 extra credit points)*

The default classification threshold is 0.5. It is just a convention — if you raise it, the model predicts "Male" less often; if you lower it, more often. A threshold change shifts the false-positive-vs-false-negative tradeoff, and it can shift that tradeoff differently across subgroups. This cell tests what happens to the best/worst-group accuracy as the threshold sweeps from 0.3 to 0.7.


In [ ]:
# Bonus: threshold sweep

thresholds = np.round(np.arange(0.3, 0.71, 0.05), 2)

best_row  = group_metrics.loc[group_metrics["accuracy"].idxmax()]
worst_row = group_metrics.loc[group_metrics["accuracy"].idxmin()]
best_mask  = labels_df[best_row["attribute"]].values  == best_row["group"]
worst_mask = labels_df[worst_row["attribute"]].values == worst_row["group"]

acc_overall, acc_best, acc_worst, gaps = [], [], [], []
for t in thresholds:
    yp_t = (y_pred_proba >= t).astype(int)
    ao = accuracy_score(y_true, yp_t)
    ab = accuracy_score(y_true[best_mask],  yp_t[best_mask])
    aw = accuracy_score(y_true[worst_mask], yp_t[worst_mask])
    acc_overall.append(ao)
    acc_best.append(ab)
    acc_worst.append(aw)
    gaps.append(ab - aw)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresholds, acc_overall, marker="o", label="Overall", color="black")
ax.plot(
    thresholds, acc_best, marker="s", color="#2ca02c",
    label=f"Best:  {best_row['attribute']}={best_row['group']}",
)
ax.plot(
    thresholds, acc_worst, marker="^", color="#d62728",
    label=f"Worst: {worst_row['attribute']}={worst_row['group']}",
)
ax.axvline(0.5, linestyle="--", color="gray", alpha=0.5,
           label="Default threshold (0.5)")
ax.set_xlabel("Classification threshold")
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy vs. threshold by subgroup")
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

gaps_arr = np.abs(np.array(gaps))
min_gap_idx = int(np.argmin(gaps_arr))
default_idx = int(np.argmin(np.abs(thresholds - 0.5)))

print(f"Threshold that minimizes |best - worst| gap: {thresholds[min_gap_idx]:.2f}")
print(f"  Gap there:      {gaps[min_gap_idx]:+.3f}")
print(f"  Gap at t=0.5:   {gaps[default_idx]:+.3f}")
print(f"  Overall acc at t={thresholds[min_gap_idx]:.2f}: {acc_overall[min_gap_idx]:.3f}")
print(f"  Overall acc at t=0.5:   {acc_overall[default_idx]:.3f}")

print("\nTakeaway: a different threshold may reduce the gap between subgroups")
print("at the cost of overall accuracy. There is no single correct threshold —")
print("the choice depends on what the deployer values.")


---
## Demo Prep

Your Module 5 demo is a **three- to five-minute** live walkthrough in class. Your notebook is your demo. Practice once end-to-end before class: flow from Cell 3 (overall metrics) → Cell 7 (scorecard) → Cell 8 (failure gallery), with your live modification in between.

The demo rubric awards 25 points for the live modification. Good options:

- Change the classification threshold and rerun Cells 3–7.
- Swap the slicing attribute in Cell 4 (e.g., add `Bald` or `Heavy_Makeup`).
- Change the intersectional pair in Cell 5 and compare.

Pick one, know what you expect to happen, and be ready to explain the result.
